# Lab 3 - Regression

**Learning goals**
- Fit and interpret linear and multiple linear regression.
- Visualize data and a fitted regression line.
- Evaluate models with Train/Test split, MSE and R².
- Reason about **assumptions** and **limitations** of linear regression.

In this lab you will use Mentimeter to answer some questions, to make it a bit more interactive.
Scan this QR code to get into the correct Mentimeter page:

<p align="center">
    <img src="pics/menti_QR_code.png" alt="QR code to access mentimeter" width="200">
</p>

 
Alternatively, you can go to the website: https://www.mentimeter.com and use the access code 6898 1743


# To run this lab you will need a virtual environment (venv) with the following packages installed:

- Pandas
- Numpy
- matplotlib
- scikit-learn
- mpl_toolkits (only for the 3d plot, but you can skip this and just look at it on the big screen)
- plotly (same)
- Jupyter

To create a virtual environment, you should:
- Open the folder where this file is, using visual studio
- Open a new terminal
- Run this command: python -m venv [name of your venv].

Once the venv is created, you can use pip to install the packages.
For example:

pip install pandas numpy scikit-learn matplotlib jupyter

this will take care of the installation process for you, and while these commands run, you can take a look at the first part of the lab, which is a shallow recap of the previous lecture.

# Part 0 — Intuition with Height & Weight, theory refresh

We start with a familiar dataset: **Height (inches)** and **Weight (pounds)**.

First, we will make sure that we can actually understand what we are talking about, going away from the freedom units in the dataset and using centimeters and kilograms.

Then we will start with the regression part: we will (1) fit a simple linear regression, (2) visualize slope & intercept, (3) try different parameters to see error changes, and (4) show a **nonlinear** pitfall with BMI.

In [ ]:
# Setup: imports and data loading

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

HW_PATH = "data/SOCR-HeightWeight.csv"
df_hw = pd.read_csv(HW_PATH)
df_hw.head()

In [ ]:
# Convert Weight from Pounds to Kilograms and Height from Inches to Centimeters

df_hw['Weight(kg)'] = df_hw['Weight(Pounds)'] * 0.453592
df_hw['Height(m)'] = df_hw['Height(Inches)'] * 0.0254

df_hw[['Height(Inches)', 'Height(m)', 'Weight(Pounds)', 'Weight(kg)']].head()

### Theory refresh: how are the slope and intercept calculated?

When we fit a simple linear regression model:

$$
y = \beta_0 + \beta_1 x
$$

we want to find the **slope** $\beta_1$ and the **intercept** $\beta_0$ that minimize the sum of squared errors between the predicted and observed values.

---

#### Slope

The slope is calculated as:

$$
\beta_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}
$$

- This is the **covariance** between $x$ and $y$, divided by the **variance** of $x$.
- Intuitively, it measures how much $y$ changes on average for a one-unit change in $x$.

---

#### Intercept

Once the slope is known, the intercept is:

$$
\beta_0 = \bar{y} - \beta_1 \bar{x}
$$

- This ensures the regression line passes through the point $(\bar{x}, \bar{y})$.
- Geometrically, it is the value of $y$ where the line crosses the vertical axis (when $x=0$).

---

#### Multiple regression (matrix form)

For multiple predictors (features), we write the model as:

$$
\hat{y} = X \beta
$$

where:
- $X$ is the data matrix ($n \times p$, with an extra column of 1’s for the intercept),
- $\beta$ is the vector of coefficients.

The solution is:

$$
\hat{\beta} = (X^T X)^{-1} X^T y
$$

This is the matrix generalization of the slope/intercept formulas.


### Calculating the slope manually

The slope formula uses the summation symbol $\sum$, which means “add up over all data points.”  

In words:

- For each data point $(x_i, y_i)$, compute how far $x_i$ is from the mean $\bar{x}$ and how far $y_i$ is from the mean $\bar{y}$.  
- Multiply these deviations together, and **sum** them across all points to get the **numerator**.  
- Then, compute the squared deviations of $x_i$ from $\bar{x}$, and **sum** them across all points to get the **denominator**.  
- Divide numerator by denominator to get the slope.

Formally:

$$
\beta_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}
$$

Scary looking formula, with greek letters! Luckily, it's way easier to see it as code:


In [ ]:
# Manual calculation of slope:

x_mean = df_hw['Height(m)'].mean()
y_mean = df_hw['Weight(kg)'].mean()

num, den = 0, 0
for i in range(len(df_hw)):
    num += (df_hw['Height(m)'][i] - x_mean) * (df_hw['Weight(kg)'][i] - y_mean)
    den += (df_hw['Height(m)'][i] - x_mean) ** 2
slope = num / den

print(f"Slope: {slope:.2f} kg/m")

### Calculating the intercept manually

Once we know the slope $\beta_1$, the intercept $\beta_0$ can be calculated with a very simple formula:

$$
\beta_0 = \bar{y} - \beta_1 \bar{x}
$$

In words:

- Take the **mean of $y$** ($\bar{y}$).  
- Subtract the slope $\beta_1$ times the **mean of $x$** ($\bar{x}$).  
- The result is the intercept $\beta_0$.  

This ensures that the regression line passes exactly through the point $(\bar{x}, \bar{y})$ — the average of the data.

In [ ]:
# Manual calculation of intercept:

intercept = y_mean - slope * x_mean
print(f"Intercept: {intercept:.2f} kg")

What does this tell us? Someone who is 0 cm tall should have a negative weight of 37.46 kg, which is at least... curious.

### Let's now use the power of Machine Learning to solve all of our problems!
<p align="center">
    <img src="pics/power.jpg" alt="Power">
</p>

Unsurprisingly, someone has worked on automating all of this before. There are libraries out there which solve our problems in one call: let's see how they work.

In [ ]:
# Simple linear regression: Weight ~ Height
X_hw = df_hw[['Height(m)']].values
y_hw = df_hw['Weight(kg)'].values

# Correlation
corr_hw = np.corrcoef(X_hw.T, y_hw)[0, 1]
print("Correlation between Height and Weight:", round(float(corr_hw), 4))

lin_hw = LinearRegression().fit(X_hw, y_hw)
slope_hw = float(lin_hw.coef_[0])
intercept_hw = float(lin_hw.intercept_)
print("Best-fit line: Weight ≈ {:.2f} * Height + {:.2f}".format(slope_hw, intercept_hw))

# How well does the model fit the data? we can use R^2 to see how much of the variance in the data is explained by the model.
r2_hw = lin_hw.score(X_hw, y_hw)
print("R^2: {:.2f}".format(r2_hw))

X_range = np.linspace(X_hw.min(), X_hw.max(), 200).reshape(-1, 1)
y_pred_line = lin_hw.predict(X_range)

plt.figure()
plt.scatter(X_hw, y_hw, alpha=0.3, label="Data")
plt.plot(X_range, y_pred_line, linewidth=2, label="Fitted line", color="green")
plt.xlabel("Height (m)")
plt.ylabel("Weight (kg)")
plt.title("Simple Linear Regression: Height vs Weight")
plt.legend()
plt.show()

We found the same slope and intercept! what happens if we try to change these numbers a bit? Does the error change at all?

In [ ]:
candidates = [
    (30, -2.0),
    (50, -24.0),
    (slope_hw, intercept_hw),  # OLS best-fit
    (60, -49.0)
]

plt.figure()
plt.scatter(X_hw, y_hw, alpha=0.3, label="Data")

X_line = np.linspace(X_hw.min(), X_hw.max(), 200).reshape(-1,1)

for s, b in candidates:
    y_line = s * X_line + b
    rmse = root_mean_squared_error(y_hw, (s * X_hw + b).ravel())
    plt.plot(X_line, y_line, linewidth=2, label=f"slope={s:.2f}, int={b:.1f}, RMSE={rmse:.1f}")

plt.xlabel("Height (m)")
plt.ylabel("Weight (kg)")
plt.title("Different Regression Lines and Their Errors")
plt.legend()
plt.show()

We can visualize the four linear regression lines from the previous example zooming out a bit, to see the different intercepts and slopes:


<img src="pics/animated_fit.gif" alt="Animated regression fit">


### Question time!
Look at the picture below and use mentimeter to answer the questions.

<img src="pics/annotated_regression.jpg" alt="Annotated regression picture">


In [ ]:
df_hw['BMI'] = df_hw['Weight(kg)'] / (df_hw['Height(m)'] ** 2)

X_H = df_hw[['Height(m)']].values
X_W = df_hw[['Weight(kg)']].values
y_bmi = df_hw['BMI'].values

lin_bmi = LinearRegression().fit(X_H, y_bmi)
r2_bmi = lin_bmi.score(X_H, y_bmi)

lin_bmi_W = LinearRegression().fit(X_W, y_bmi)
r2_bmi_W = lin_bmi_W.score(X_W, y_bmi)
print("R^2 for BMI ~ Height (linear):", round(float(r2_bmi), 4))
print("R^2 for BMI ~ Weight (linear):", round(float(r2_bmi_W), 4))

Xb = np.linspace(X_H.min(), X_H.max(), 200).reshape(-1,1)
yb = lin_bmi.predict(Xb)

plt.figure()
plt.scatter(X_H, y_bmi, alpha=0.3, label="BMI data")
plt.plot(Xb, yb, linewidth=2, label="Linear fit")
plt.xlabel("Height (m)")
plt.ylabel("BMI")
plt.title("Trying to predict BMI from Height (nonlinearity)")
plt.legend()
plt.show()

### Multiple linear regression: Predict BMI from Height & Weight

Now we try a **multiple linear regression** using both **Height(m)** and **Weight(kg)** as inputs to predict **BMI**.
We'll show a 3D scatter of the data and overlay the model's prediction surface.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D

X_multi = df_hw[['Height(m)', 'Weight(kg)']].values
y_bmi = df_hw['BMI'].values

lin_multi = LinearRegression().fit(X_multi, y_bmi)
y_pred_multi = lin_multi.predict(X_multi)

print("Multiple Linear Regression (BMI ~ Height + Weight)")
print("Coefficients [Height(m), Weight(kg)]:", lin_multi.coef_)
print("Intercept:", lin_multi.intercept_)
print("R^2:", round(float(r2_score(y_bmi, y_pred_multi)), 4))
print("RMSE:", round(float(root_mean_squared_error(y_bmi, y_pred_multi)), 4))

# 3D scatter + plane
H = X_multi[:, 0]
W = X_multi[:, 1]

# Create a grid
H_lin = np.linspace(H.min(), H.max(), 30)
W_lin = np.linspace(W.min(), W.max(), 30)
HH, WW = np.meshgrid(H_lin, W_lin)
ZZ = lin_multi.predict(np.c_[HH.ravel(), WW.ravel()]).reshape(HH.shape)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(H, W, y_bmi, alpha=0.2)
ax.plot_surface(HH, WW, ZZ, alpha=0.6, linewidth=0)

ax.set_xlabel("Height (m)")
ax.set_ylabel("Weight (kg)")
ax.set_zlabel("BMI")
ax.set_title("3D: BMI ~ Height + Weight (Linear Regression)")
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=H, y=W, z=y_bmi,
    mode='markers',
    opacity=0.4,
    marker=dict(size=3),
    name='BMI data'
))
fig.add_trace(go.Surface(
    x=HH, y=WW, z=ZZ,
    opacity=0.7,
    showscale=False,
    name='Regression plane'
))
fig.update_layout(
    scene=dict(
        xaxis_title='Height (m)',
        yaxis_title='Weight (kg)',
        zaxis_title='BMI',
    ),
    title='Interactive 3D: BMI ~ Height + Weight (Linear Regression)',
    margin=dict(l=0, r=0, t=40, b=0),
    height=600
)
fig.show()


## It's your turn now!

## 0. Setup & Context (Credit Card Balance)

We will use the **Credit Card Balance** dataset (from the ISL book). The columns are:

- Income: Income in thousands of $
- Limit: Credit limit
- Rating: Credit rating
- Cards: Number of credit cards
- Age: Age in years
- Education: Education in years
- Own: A factor with levels No and Yes indicating whether the individual owns a home
- Student: A factor with levels No and Yes indicating whether the individual is a student
- Married: A factor with levels No and Yes indicating whether the individual is married
- Region: A factor with levels East, South, and West indicating the individual’s geographical location
- Balance: Average credit card balance in $


The obvious target in this dataset is the balance column.

In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

DATA_PATH = "data/Credit.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## Part A — Warm-up: Explore & Visualize

**Task A1** Choose one numeric feature `X` and
**Task A2** plot it against the target `y`.  
**Task A3 (Correlation):** Compute the correlation between your `X` and `y`.

In [ ]:
y = df['...'].values                                                                                # which feature will you choose as a target?
candidate_features = [c for c in df.columns if c != '...']
chosen = candidate_features[...]                                                                    # which feature will you choose as a predictor?
X = df[chosen]

# Create your plot here. A similar one has been created in the initial example!

corr = np.corrcoef(X, y)[0, 1]
print("Correlation between", chosen, "and target:", round(float(corr), 4))

## Part B — Simple Linear Regression

**Task B1 (Fit):** Fit a **simple** linear regression: `y ~ X`.  
**Task B2 (Interpret):** Print the coefficient and intercept.  
**Task B3 (Line overlay):** Plot the fitted line over the scatter from Part A.

In [ ]:
X_2d = X.values.reshape(-1, 1)

lin = LinearRegression()
lin.fit(X_2d, y)

print("Coefficient (slope):", float(lin.coef_[0]))
print("Intercept:", float(lin.intercept_))

x_sorted_idx = np.argsort(X.values)
X_sorted = X.values[x_sorted_idx]
y_pred_sorted = lin.predict(X.values.reshape(-1, 1))[x_sorted_idx]

plt.figure()
plt.scatter(X, y, alpha=0.7, label="data")
plt.plot(X_sorted, y_pred_sorted, linewidth=2, label="fitted line")
plt.xlabel(f"X ({chosen})")
plt.ylabel("Target")
plt.title("Simple Linear Regression fit")
plt.legend()
plt.show()

## Part C — Multiple Linear Regression

**Task C1 (Train/Test):** Choose a set of features `X_multi` (2–6 columns) and keep the same target `y`.  
**Task C2 (Fit & Evaluate):** Compute **RMSE** and **R²** on train/test.  
**Task C3 (Coefficients):** Print coefficients with names; which features are most influential?

In [ ]:
exclude_cols = ['...']                                                                  # which feature(s) will you exclude? most likely the target itself, but maybe some others too
X_multi = df.drop(columns=exclude_cols)

y = df['...']                                                                           # keep the same target!

X_train, X_test, y_train, y_test = None, None, None, None                               # split the data into training and test sets

# Create a LinearRegression model and fit it to the training data

y_pred_train = None                                                                     # predict on the training set      
y_pred_test = None                                                                      # predict on the test set

# Compute the metrics: an error metric (MSE or RMSE) and R^2 for both training and test sets
mse_train = None
mse_test = None
r2_train = None
r2_test = None

print("Train MSE:", float(mse_train))
print("Test  MSE:", float(mse_test))
print("Train R^2:", float(r2_train))
print("Test  R^2:", float(r2_test))

# print the coefficients of the model, sorted by their absolute value

## Part D — Assumptions & Diagnostics

**Task D1 (Residual plot)**  
**Task D2 (Nonlinearity):** Add a squared term for top feature; re-evaluate.  
**Task D3 (Reflection):** Note one limitation of this approach.

In [ ]:
residuals = None  # A very important part of regression analysis is to check the residuals (errors) of the model. compute them here!

# and now, plot the residuals vs the predicted values

# if linear regression does not work, sometimes adding polynomial features can help
# add a squared term for the feature with the largest absolute coefficient!
top_feature = None                                      # which feature has the largest absolute coefficient?
X_train_ext = X_train.copy()                            # it's always safer to work on copies of the data when you are modifying the original features
X_test_ext = X_test.copy()
X_train_ext[top_feature + "^2"] = None                  # add the squared term for the top feature
X_test_ext[top_feature + "^2"] = None                   # same for the test set: if you add a feature to the training set, you must also add it to the test set!

reg2 = None                                             # create a new LinearRegression model. Why not the same one as before?
# fit it to the extended training data

y_pred_test_ext = None                                  # predict on the extended test set
rmse_test = None                                        # compute the RMSE on the original test set
rmse_test_ext = None                                    # compute the RMSE on the extended test set
r2_test_ext = None                                      # compute R^2 on the extended test set. This way we can check if it improved adding this polynomial feature!

print("Original test RMSE:", float(rmse_test))
print("With squared feature, test RMSE:", float(rmse_test_ext))
print("Original test R^2:", float(r2_test))
print("With squared feature, test R^2:", float(r2_test_ext))

# Part E - Regularization

In this extension, we'll explore:
- **Ridge** (L2) and
- **Lasso** (L1) regression.

They combat overfitting and handle collinearity.

> Tip: Regularization works best when features are on comparable scales, so we use a **StandardScaler**.

In [ ]:
# Imports for Part E
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# We assume X_train, X_test, y_train, y_test from Part C already exist.
# If you jump directly here, re-run Part C cells first.

## Ridge & Lasso with Cross-Validation

**Tasks:**
1. Fit **RidgeCV** (search over alphas) in a pipeline with StandardScaler.
2. Fit **LassoCV** likewise.
3. Compare **R²** and **MSE** on the **test** set vs. OLS from Part C.
4. Inspect coefficients to see **shrinkage** and (for Lasso) **sparsity**.

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]                 # you can modify this list if you want to try different alpha values for Ridge/Lasso

ridge_pipe = Pipeline([                                # Pipelines are a convenient way to chain preprocessing steps and the model together, and to be consistent between different models.
    # your pipeline should contain a StandardScaler step and a RidgeCV step, but in what order?
])
ridge_pipe.fit(None, None)                             # you can now fit the pipeline like you would fit any model
ridge_y_pred = None                                    # and you can use it to predict on the test set
ridge_rmse = None                                      # and even compute metrics on those predictions
ridge_r2 = None

lasso_pipe = Pipeline([
    # the procedure for lasso will be quite similar
])
# fit, predict, compute metrics as for ridge

# Get ridge and lasso alpha* values for the best models
ridge_alpha = getattr(ridge_pipe.named_steps['ridge'], 'alpha_', None) # this is a nice way to get the best alpha value chosen for Ridge using the pipeline (and, in my file, using Cross-validation)
lasso_alpha = getattr(lasso_pipe.named_steps['lasso'], 'alpha_', None) # if you are curious, you can change the list of alphas and see how the chosen alpha* changes.
                                                                       # alpha in this context is what was called lambda in the lecture slides: it represents the strength of regularization

print("Ridge  -> alpha*:", ridge_alpha)                                # we can now print the best alpha* values chosen for Ridge and Lasso,
print("Test R^2:", round(float(ridge_r2), 4), " | Test RMSE:", round(float(ridge_rmse), 3)) # along with the test metrics

print("Lasso  -> alpha*:", lasso_alpha)
print("Test R^2:", round(float(lasso_r2), 4), " | Test RMSE:", round(float(lasso_rmse), 3))

# Compare to OLS from Part C (requires r2_test/mse_test variables)
print("\nOLS    -> \nTest R^2:", round(float(r2_test), 4), " | Test RMSE:", round(float(rmse_test), 3)) # OLS does not have any regularization,
                                                                                                        # but we can compare its metrics to Ridge and Lasso as well.

### Coefficient comparison

Below we compare OLS vs. Ridge vs. Lasso coefficients. For Ridge/Lasso, we recover
the coefficients by fitting on the **scaled** training data and mapping back.

In [ ]:
# Refit simple models to extract comparable coefficients
ols = LinearRegression().fit(X_train, y_train)
ols_coefs = ols.coef_

# For Ridge/Lasso, we can transform X with the scaler to see effect on original scale
scaler = StandardScaler().fit(X_train)
Xtr_s = scaler.transform(X_train)

from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=getattr(ridge_pipe.named_steps['ridge'], 'alpha_', 1.0))
ridge.fit(Xtr_s, y_train)
lasso = Lasso(alpha=getattr(lasso_pipe.named_steps['lasso'], 'alpha_', 0.1), max_iter=5000)
lasso.fit(Xtr_s, y_train)

# Map scaled coefficients back to original feature scale
scale_ = scaler.scale_
ridge_coefs = ridge.coef_ / scale_
lasso_coefs = lasso.coef_ / scale_

coef_df = pd.DataFrame({
    "OLS": ols_coefs,
    "Ridge": ridge_coefs,
    "Lasso": lasso_coefs,
}, index=X_train.columns).sort_values(by="OLS", key=lambda s: s.abs(), ascending=False)

coef_df

# End of Lab 3

Great job on surviving this session!

<p align="center">
    <img src="pics/well_done.jpg" alt="End of Lab 3" width="300">
</p>